# 🦀 Day 7: Mini-Project — Multi-Threaded Web Scraper Skeleton

---

## Project Overview

We'll build a **concurrent URL fetcher** that:
- Uses a thread pool for work distribution
- Maintains a URL queue with `Arc<Mutex<Vec<String>>>`
- Simulates HTTP fetching (no real HTTP in notebooks)
- Collects results via channels

This combines threads, channels, `Arc`, and `Mutex` from this week!

## Step 1: Data Structures and Simulated Fetch

In [ ]:
use std::sync::mpsc;
use std::sync::{Arc, Mutex};
use std::thread;
use std::time::Duration;

#[derive(Debug, Clone)]
struct FetchResult {
    url: String,
    status: u16,
    body_len: usize,
}

// Simulated HTTP fetch — in a real app, use reqwest or similar
fn fetch_url(url: &str) -> FetchResult {
    thread::sleep(Duration::from_millis(50)); // Simulate network delay
    FetchResult {
        url: url.to_string(),
        status: 200,
        body_len: url.len() * 10, // Mock body length
    }
}

let result = fetch_url("https://example.com");
println!("Simulated fetch: {:?}", result);

## Step 2: URL Queue with Arc<Mutex<Vec<String>>>

In [ ]:
use std::sync::{Arc, Mutex};

type UrlQueue = Arc<Mutex<Vec<String>>>;

fn create_queue(urls: Vec<String>) -> UrlQueue {
    Arc::new(Mutex::new(urls))
}

fn pop_url(queue: &UrlQueue) -> Option<String> {
    let mut q = queue.lock().unwrap();
    if q.is_empty() { None } else { Some(q.remove(0)) }
}

let urls = vec![
    "https://a.com".to_string(),
    "https://b.com".to_string(),
    "https://c.com".to_string(),
];
let queue = create_queue(urls);
println!("Popped: {:?}", pop_url(&queue));
println!("Popped: {:?}", pop_url(&queue));
println!("Popped: {:?}", pop_url(&queue));
println!("Popped: {:?}", pop_url(&queue)); // None

## Step 3: Thread Pool + Channels for Work Distribution

In [ ]:
use std::sync::mpsc;
use std::sync::{Arc, Mutex};
use std::thread;
use std::time::Duration;

#[derive(Debug, Clone)]
struct FetchResult { url: String, status: u16, body_len: usize }

fn fetch_url(url: &str) -> FetchResult {
    thread::sleep(Duration::from_millis(30));
    FetchResult { url: url.to_string(), status: 200, body_len: url.len() * 10 }
}

type UrlQueue = Arc<Mutex<Vec<String>>>;
fn pop_url(queue: &UrlQueue) -> Option<String> {
    let mut q = queue.lock().unwrap();
    if q.is_empty() { None } else { Some(q.remove(0)) }
}

let urls: Vec<String> = (1..=6).map(|i| format!("https://site{}.com", i)).collect();
let queue = Arc::new(Mutex::new(urls));
let (tx, rx) = mpsc::channel();

let num_workers = 3;
let mut handles = vec![];

for _ in 0..num_workers {
    let q = Arc::clone(&queue);
    let sender = mpsc::Sender::clone(&tx);
    handles.push(thread::spawn(move || {
        loop {
            let url = match pop_url(&q) { Some(u) => u, None => break };
            let result = fetch_url(&url);
            sender.send(result).unwrap();
        }
    }));
}

drop(tx); // Close sender so receiver knows no more messages

let results: Vec<FetchResult> = rx.iter().collect();
for h in handles { h.join().unwrap(); }

println!("Fetched {} URLs:", results.len());
for r in &results { println!("  {} -> {} ({} bytes)", r.url, r.status, r.body_len); }

## Step 4: Putting It Together — Scraper Runner

In [ ]:
use std::sync::mpsc;
use std::sync::{Arc, Mutex};
use std::thread;
use std::time::Instant;

// Uses FetchResult and fetch_url from previous cells
fn run_scraper(urls: Vec<String>, num_workers: usize) -> Vec<FetchResult> {
    let queue = Arc::new(Mutex::new(urls));
    let (tx, rx) = mpsc::channel();
    let mut handles = vec![];

    for _ in 0..num_workers {
        let q = Arc::clone(&queue);
        let sender = mpsc::Sender::clone(&tx);
        handles.push(thread::spawn(move || {
            loop {
                let url = { let mut g = q.lock().unwrap(); if g.is_empty() { break } g.remove(0) };
                sender.send(fetch_url(&url)).unwrap();
            }
        }));
    }
    drop(tx);

    let results: Vec<FetchResult> = rx.iter().collect();
    for h in handles { h.join().unwrap(); }
    results
}

let urls: Vec<String> = (1..=8).map(|i| format!("https://page{}.example.com", i)).collect();
let start = Instant::now();
let results = run_scraper(urls, 4);
println!("Scraped {} URLs in {:?}", results.len(), start.elapsed());
println!("Sample: {:?}", &results[..2]);

## 🏋️ Exercises

In [ ]:
// Exercise: Extend the scraper with one of:
// A) Retry logic — if fetch "fails" (simulate by random status 500), retry up to 2 times
// B) Rate limiting — use a shared AtomicUsize or Mutex to limit fetches per second
//
// For (A): modify fetch_url to sometimes return status 500, and retry in the worker loop
// For (B): add a simple "max concurrent" or sleep between fetches

use std::sync::mpsc;
use std::sync::{Arc, Mutex};
use std::thread;
use std::time::Duration;

// YOUR CODE HERE


## 🎯 Key Takeaways

1. **Thread pool + channels** — workers pull from a shared queue, send results via channel
2. **URL queue** — `Arc<Mutex<Vec<String>>>` for thread-safe work distribution
3. **Simulated fetch** — use `thread::sleep` and mock data when real HTTP isn't available
4. **Results collection** — `rx.iter().collect()` gathers all worker output
5. **`drop(tx)`** — closing the sender signals workers that no more work is coming
6. **Week 13 Complete!** You've built threads, channels, Mutex, Arc, Rayon, atomics, and a scraper skeleton.

---

**Next week:** Async/Await — the future of concurrent Rust! ⚡